# Ranking Model - F1 Podium Prediction

Model Learning to Rank untuk mengoptimalkan urutan podium dalam setiap race.

## Model:
1. LGBMRanker (LambdaRank)
2. XGBRanker (rank:ndcg)
3. Two-Stage Model (Classifier + Ranker)

## Relevance Score:
- P1 (positionOrder=1): 3
- P2 (positionOrder=2): 2
- P3 (positionOrder=3): 1
- Non-podium: 0

## Split:
- Train: 2014-2022
- Validation: 2023
- Test: 2024-2025

## Metrik:
- NDCG@3 (metric utama ranking)
- Podium Hit Rate
- Winner Accuracy
- Exact Podium Drivers

In [ ]:
import pandas as pd
import numpy as np
import warnings
import joblib
import os
warnings.filterwarnings('ignore')

print('Library loaded.')

## 1. Load Dataset

In [ ]:
# Load dataset
df = pd.read_parquet('../data/processed/model_dataset.parquet')
print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.shape[1]}')
print(f'Years: {df["year"].min()} - {df["year"].max()}')
print(f'Unique races: {df["raceId"].nunique()}')

In [ ]:
# Definisikan fitur (sama dengan notebook 06)
id_columns = ['raceId', 'driverId', 'constructorId', 'year', 'round',
              'circuitId', 'date', 'race_name', 'driverRef', 'team',
              'positionOrder', 'is_podium']

feature_cols = [c for c in df.columns if c not in id_columns]
print(f'Jumlah fitur: {len(feature_cols)}')

## 2. Relevance Score & Data Split

Buat relevance score untuk ranking: P1=3, P2=2, P3=1, non-podium=0.

In [ ]:
# Buat relevance score
df['relevance'] = df['positionOrder'].apply(
    lambda x: 3 if x == 1 else (2 if x == 2 else (1 if x == 3 else 0))
)

print('Distribusi relevance:')
print(df['relevance'].value_counts().sort_index())

In [ ]:
# Time-based split
train_mask = df['year'] <= 2022
val_mask = df['year'] == 2023
test_mask = df['year'] >= 2024

X_train = df[train_mask][feature_cols].copy()
y_train = df[train_mask]['is_podium'].copy()
rel_train = df[train_mask]['relevance'].copy()

X_val = df[val_mask][feature_cols].copy()
y_val = df[val_mask]['is_podium'].copy()
rel_val = df[val_mask]['relevance'].copy()

X_test = df[test_mask][feature_cols].copy()
y_test = df[test_mask]['is_podium'].copy()
rel_test = df[test_mask]['relevance'].copy()

# Simpan metadata untuk evaluasi per race
train_meta = df[train_mask][['raceId', 'year', 'round', 'driverId', 'constructorId',
                              'driverRef', 'team', 'positionOrder', 'is_podium', 'relevance']].copy()
val_meta = df[val_mask][['raceId', 'year', 'round', 'driverId', 'constructorId',
                          'driverRef', 'team', 'positionOrder', 'is_podium', 'relevance']].copy()
test_meta = df[test_mask][['raceId', 'year', 'round', 'driverId', 'constructorId',
                            'driverRef', 'team', 'positionOrder', 'is_podium', 'relevance']].copy()

print(f'Train   : {X_train.shape}')
print(f'Val     : {X_val.shape}')
print(f'Test    : {X_test.shape}')

## 3. Handling Missing Values

Isi missing values dengan median dari train set (sama seperti notebook 06).

In [ ]:
# Cek missing values
missing = X_train.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f'Fitur dengan missing values di train: {len(missing)}')
if len(missing) > 0:
    display(missing.head(20))

In [ ]:
# Impute dengan median dari train
median_values = X_train.median()

X_train = X_train.fillna(median_values)
X_val = X_val.fillna(median_values)
X_test = X_test.fillna(median_values)

print(f'Missing setelah impute:')
print(f'  Train: {X_train.isna().sum().sum()}')
print(f'  Val: {X_val.isna().sum().sum()}')
print(f'  Test: {X_test.isna().sum().sum()}')

## 4. Group Size untuk Ranking

Ranking model membutuhkan parameter `group` yang berisi jumlah pembalap di setiap race.

In [ ]:
# Hitung group size per raceId untuk ranking
def get_group_sizes(df_data, meta_data):
    """Dapatkan array group sizes dari metadata sesuai urutan data."""
    race_counts = meta_data.groupby('raceId').size()
    return race_counts.values

group_train = get_group_sizes(X_train, train_meta)
group_val = get_group_sizes(X_val, val_meta)
group_test = get_group_sizes(X_test, test_meta)

print(f'Train groups: {len(group_train)} races, total: {group_train.sum()} rows')
print(f'Val groups: {len(group_val)} races, total: {group_val.sum()} rows')
print(f'Test groups: {len(group_test)} races, total: {group_test.sum()} rows')
print(f'\nSample group sizes: {group_train[:10]}')

## 5. Evaluation Functions

Fungsi evaluasi untuk ranking model (NDCG@3, podium hit rate, dll).

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             log_loss, brier_score_loss)

def evaluate_proba(y_true, y_prob, label='Model'):
    """Evaluasi probabilitas"""
    results = {
        'Model': label,
        'ROC-AUC': roc_auc_score(y_true, y_prob),
        'PR-AUC': average_precision_score(y_true, y_prob),
        'Log Loss': log_loss(y_true, y_prob),
        'Brier Score': brier_score_loss(y_true, y_prob)
    }
    return results


def evaluate_ranking_per_race(df_meta, scores, label='Model', higher_is_better=True):
    """
    Evaluasi ranking per race.
    scores: array of ranking scores (higher = better candidate for podium)
    """
    df_meta = df_meta.copy()
    df_meta['score'] = scores
    
    race_results = []
    
    for race_id, group in df_meta.groupby('raceId'):
        if len(group) < 3:
            continue
        
        # Aktual
        actual_podium = group[group['positionOrder'] <= 3]['driverRef'].tolist()
        actual_p1 = group[group['positionOrder'] == 1]['driverRef'].values
        
        if len(actual_podium) < 3 or len(actual_p1) == 0:
            continue
        
        # Prediksi: urutkan berdasarkan score
        if higher_is_better:
            top3 = group.nlargest(3, 'score')
        else:
            top3 = group.nsmallest(3, 'score')
        
        predicted_podium = top3['driverRef'].tolist()
        predicted_p1 = top3.iloc[0]['driverRef']
        
        # Relevance scores untuk NDCG
        group = group.copy()
        group['relevance'] = group['positionOrder'].apply(
            lambda x: 3 if x == 1 else (2 if x == 2 else (1 if x == 3 else 0))
        )
        
        # Predicted ranking
        if higher_is_better:
            predicted = group.sort_values('score', ascending=False)
        else:
            predicted = group.sort_values('score', ascending=True)
        predicted_rel = predicted.head(3)['relevance'].tolist()
        
        # NDCG@3
        dcg = sum((2**rel - 1) / np.log2(i + 2) for i, rel in enumerate(predicted_rel[:3]))
        ideal_rel = sorted(group['relevance'].tolist(), reverse=True)[:3]
        idcg = sum((2**rel - 1) / np.log2(i + 2) for i, rel in enumerate(ideal_rel))
        ndcg = dcg / idcg if idcg > 0 else 0
        
        hits = sum(1 for d in actual_podium if d in predicted_podium)
        exact_drivers = all(d in predicted_podium for d in actual_podium)
        winner_correct = actual_p1[0] == predicted_p1
        
        race_results.append({
            'raceId': race_id,
            'year': group['year'].iloc[0],
            'race_name': group['race_name'].iloc[0],
            'podium_hits': hits,
            'exact_drivers': int(exact_drivers),
            'winner_correct': int(winner_correct),
            'ndcg': ndcg,
            'n_drivers': len(group)
        })
    
    results_df = pd.DataFrame(race_results)
    
    total_races = len(results_df)
    total_hits = results_df['podium_hits'].sum()
    possible_hits = total_races * 3
    
    print(f'=== {label} ===')
    print(f'Total races: {total_races}')
    print(f'Podium Hit Rate: {total_hits}/{possible_hits} = {total_hits/possible_hits:.4f}')
    print(f'Exact Podium Drivers: {results_df["exact_drivers"].mean():.4f}')
    print(f'Winner Accuracy: {results_df["winner_correct"].mean():.4f}')
    print(f'NDCG@3: {results_df["ndcg"].mean():.4f}')
    print()
    
    return results_df

## 6. LGBMRanker

LightGBM Ranker dengan objective='lambdarank'. Model ranking utama.

In [ ]:
import lightgbm as lgb
from lightgbm import LGBMRanker

lgb_ranker = LGBMRanker(
    objective='lambdarank',
    boosting_type='gbdt',
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# Latih model dengan group sizes
lgb_ranker.fit(
    X_train, rel_train,
    group=group_train,
    eval_set=[(X_val, rel_val)],
    eval_group=[group_val],
    eval_at=[3],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)]
)

# Prediksi score
lgb_ranker_score_val = lgb_ranker.predict(X_val)
lgb_ranker_score_test = lgb_ranker.predict(X_test)

print('LGBMRanker training selesai.')

In [ ]:
# Evaluasi LGBMRanker
lgb_ranker_val = evaluate_ranking_per_race(val_meta, lgb_ranker_score_val, 'LGBMRanker (Val)')
lgb_ranker_test = evaluate_ranking_per_race(test_meta, lgb_ranker_score_test, 'LGBMRanker (Test)')

## 7. XGBRanker

XGBoost Ranker dengan objective='rank:ndcg'.

In [ ]:
from xgboost import XGBRanker

# XGBoost Ranker membutuhkan format data khusus
# group sizes harus dikonversi ke format XGBoost
import xgboost as xgb

# Buat DMatrix untuk ranking
dtrain = xgb.DMatrix(X_train, label=rel_train)
dtrain.set_group(group_train)

dval = xgb.DMatrix(X_val, label=rel_val)
dval.set_group(group_val)

dtest = xgb.DMatrix(X_test, label=rel_test)
dtest.set_group(group_test)

params = {
    'objective': 'rank:ndcg',
    'eval_metric': 'ndcg@3',
    'eta': 0.1,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 10,
    'random_state': 42,
    'n_jobs': -1
}

xgb_ranker = xgb.train(
    params,
    dtrain,
    num_boost_round=200,
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=50,
    verbose_eval=50
)

# Prediksi score
xgb_ranker_score_val = xgb_ranker.predict(dval)
xgb_ranker_score_test = xgb_ranker.predict(dtest)

print('XGBRanker training selesai.')

In [ ]:
# Evaluasi XGBRanker
xgb_ranker_val = evaluate_ranking_per_race(val_meta, xgb_ranker_score_val, 'XGBRanker (Val)')
xgb_ranker_test = evaluate_ranking_per_race(test_meta, xgb_ranker_score_test, 'XGBRanker (Test)')

## 8. Two-Stage Model (Section 12.3)

Stage 1: Classifier predict podium probability.
Stage 2: Ranker re-rank top candidates.

In [ ]:
# Stage 1: Load classifier dari notebook 06 (atau train ulang)
from lightgbm import LGBMClassifier

lgb_clf = LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    class_weight='balanced',
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_clf.fit(X_train, y_train, eval_set=[(X_val, y_val)])

# Prediksi probabilitas podium (Stage 1)
clf_prob_test = lgb_clf.predict_proba(X_test)[:, 1]

print('Stage 1 (Classifier) selesai.')

In [ ]:
# Stage 2: Untuk setiap race, ambil ~10 kandidat teratas dari classifier,
# lalu re-rank dengan LGBMRanker

two_stage_results = []

for race_id, group in test_meta.groupby('raceId'):
    if len(group) < 3:
        continue
    
    # Ambil index baris di X_test yang sesuai dengan race ini
    race_indices = group.index
    
    # Stage 1: filter top kandidat (min 5, max 10, atau semua jika < 10)
    n_candidates = min(10, len(race_indices))
    top_k_idx = race_indices[np.argsort(clf_prob_test[race_indices])[::-1][:n_candidates]]
    
    # Stage 2: re-rank dengan ranker score
    ranker_scores_subset = lgb_ranker_score_test[top_k_idx]
    
    # Final ranking berdasarkan ranker score
    final_ranking = top_k_idx[np.argsort(ranker_scores_subset)[::-1]]
    
    # Ambil top 3
    predicted_top3 = group.loc[final_ranking[:3]]
    predicted_podium = predicted_top3['driverRef'].tolist()
    predicted_p1 = predicted_top3.iloc[0]['driverRef']
    
    # Aktual
    actual_podium = group[group['positionOrder'] <= 3]['driverRef'].tolist()
    actual_p1 = group[group['positionOrder'] == 1]['driverRef'].values
    if len(actual_podium) < 3 or len(actual_p1) == 0:
        continue
    
    # Metrik
    hits = sum(1 for d in actual_podium if d in predicted_podium)
    exact_drivers = all(d in predicted_podium for d in actual_podium)
    winner_correct = actual_p1[0] == predicted_p1
    
    two_stage_results.append({
        'raceId': race_id,
        'year': group['year'].iloc[0],
        'race_name': group['race_name'].iloc[0],
        'podium_hits': hits,
        'exact_drivers': int(exact_drivers),
        'winner_correct': int(winner_correct),
        'n_drivers': len(group)
    })

two_stage_df = pd.DataFrame(two_stage_results)

total_races = len(two_stage_df)
total_hits = two_stage_df['podium_hits'].sum()
possible_hits = total_races * 3

print('=== Two-Stage Model (Test) ===')
print(f'Total races: {total_races}')
print(f'Podium Hit Rate: {total_hits}/{possible_hits} = {total_hits/possible_hits:.4f}')
print(f'Exact Podium Drivers: {two_stage_df["exact_drivers"].mean():.4f}')
print(f'Winner Accuracy: {two_stage_df["winner_correct"].mean():.4f}')

## 9. Load Classifier Results (Baseline dari Notebook 06)

Bandingkan dengan hasil classifier dari notebook 06.

In [ ]:
# Evaluasi classifier sebagai pembanding di test set
lgb_clf_score_test = lgb_clf.predict_proba(X_test)[:, 1]
lgb_clf_test_race = evaluate_ranking_per_race(test_meta, lgb_clf_score_test, 'LightGBM Classifier (Test)')

In [ ]:
# Ambil metrik NDCG dari classifier test
clf_ndcg = lgb_clf_test_race['ndcg'].mean()
clf_podium_hit = lgb_clf_test_race['podium_hits'].sum() / (len(lgb_clf_test_race) * 3)
clf_winner = lgb_clf_test_race['winner_correct'].mean()
clf_exact = lgb_clf_test_race['exact_drivers'].mean()

## 10. Perbandingan Semua Model

Bandingkan classifier, LGBMRanker, XGBRanker, dan Two-Stage Model.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='darkgrid')

# Tabel perbandingan
comparison_data = []

models_info = [
    ('LightGBM Classifier', lgb_clf_test_race),
    ('LGBMRanker', lgb_ranker_test),
    ('XGBRanker', xgb_ranker_test),
]

for name, df_race in models_info:
    total = len(df_race)
    hits = df_race['podium_hits'].sum()
    comparison_data.append({
        'Model': name,
        'Total Races': total,
        'Podium Hit Rate': hits / (total * 3),
        'Winner Accuracy': df_race['winner_correct'].mean(),
        'Exact Drivers': df_race['exact_drivers'].mean(),
        'NDCG@3': df_race['ndcg'].mean()
    })

# Tambah Two-Stage
total_ts = len(two_stage_df)
hits_ts = two_stage_df['podium_hits'].sum()
comparison_data.append({
    'Model': 'Two-Stage (Clf+Rank)',
    'Total Races': total_ts,
    'Podium Hit Rate': hits_ts / (total_ts * 3),
    'Winner Accuracy': two_stage_df['winner_correct'].mean(),
    'Exact Drivers': two_stage_df['exact_drivers'].mean(),
    'NDCG@3': np.nan  # Two-stage tidak punya NDCG karena pakai subset
})

comparison = pd.DataFrame(comparison_data)
display(comparison)

In [ ]:
# Visualisasi perbandingan
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_labels = comparison['Model'].tolist()
x = np.arange(len(model_labels))
width = 0.3

# Podium Hit Rate & Winner Accuracy
ax = axes[0]
ax.bar(x - width/2, comparison['Podium Hit Rate'], width, label='Podium Hit Rate', color='steelblue')
ax.bar(x + width/2, comparison['Winner Accuracy'], width, label='Winner Accuracy', color='coral')
ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Podium Hit Rate & Winner Accuracy (Test Set)')
ax.set_xticks(x)
ax.set_xticklabels(model_labels, rotation=20)
ax.legend()
ax.set_ylim(0, 1)

# Exact Drivers & NDCG@3
ax = axes[1]
ndcg_values = comparison['NDCG@3'].fillna(0)
ax.bar(x - width/2, comparison['Exact Drivers'], width, label='Exact Drivers', color='seagreen')
ax.bar(x + width/2, ndcg_values, width, label='NDCG@3', color='goldenrod')
ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Exact Drivers & NDCG@3')
ax.set_xticks(x)
ax.set_xticklabels(model_labels, rotation=20)
ax.legend()

plt.tight_layout()
plt.show()

## 11. Simpan Model dan Hasil

In [ ]:
import joblib
import os

# Buat direktori
os.makedirs('../models/ranker/', exist_ok=True)
os.makedirs('../reports/metrics/', exist_ok=True)

# Simpan model
joblib.dump(lgb_ranker, '../models/ranker/lgb_ranker.pkl')
print('LGBMRanker saved: models/ranker/lgb_ranker.pkl')

joblib.dump(xgb_ranker, '../models/ranker/xgb_ranker.pkl')
print('XGBRanker saved: models/ranker/xgb_ranker.pkl')

# Simpan hasil metrik
comparison.to_csv('../reports/metrics/ranker_results.csv', index=False)
print('Ranker results saved: reports/metrics/ranker_results.csv')

# Simpan per-race results
lgb_ranker_test.to_csv('../reports/metrics/lgb_ranker_per_race.csv', index=False)
xgb_ranker_test.to_csv('../reports/metrics/xgb_ranker_per_race.csv', index=False)
print('Per-race results saved.')

In [ ]:
print('=== RINGKASAN RANKING MODEL ===')
print('=' * 65)
print(f'{"Model":<30} {"Podium Hit":<15} {"NDCG@3":<10} {"Winner":<10}')
print('-' * 65)
for _, row in comparison.iterrows():
    ndcg = f'{row["NDCG@3"]:.4f}' if not pd.isna(row['NDCG@3']) else 'N/A'
    print(f'{row["Model"]:<30} {row["Podium Hit Rate"]:<15.4f} {ndcg:<10} {row["Winner Accuracy"]:<10.4f}')
print('=' * 65)